# Model Implementation
**Project:** Employee Attrition Prediction in the Saudi Private Sector

This notebook compares several baseline classification models using the two preprocessed datasets created in the preprocessing stage. It reports initial evaluation metrics and selects the most suitable initial model for the project.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

In [3]:
def resolve_path(filename: str) -> Path:
    candidates = [
        Path.cwd() / filename,
        Path.cwd().parent / filename,
        Path.home() / 'Downloads' / filename,
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f'Could not locate {filename}. Checked: {candidates}')

tree_path = resolve_path('processed_dataset_tree.csv')
nontree_path = resolve_path('processed_dataset_nontree.csv')

df_tree = pd.read_csv(tree_path)
df_nontree = pd.read_csv(nontree_path)

print('Tree dataset:', tree_path, df_tree.shape)
print('Non-tree dataset:', nontree_path, df_nontree.shape)

Tree dataset: c:\Users\j4mai\source\repos\DS Assignment 2\processed_dataset_tree.csv (1191, 35)
Non-tree dataset: c:\Users\j4mai\source\repos\DS Assignment 2\processed_dataset_nontree.csv (1191, 190)


## Dataset Overview
We use the preprocessed outputs from the earlier stage:

- `processed_dataset_tree.csv` for tree-based models.
- `processed_dataset_nontree.csv` for non-tree models.

In [4]:
summary = pd.DataFrame(
    {
        'Dataset': ['Tree-based', 'Non-tree-based'],
        'Rows': [df_tree.shape[0], df_nontree.shape[0]],
        'Columns': [df_tree.shape[1], df_nontree.shape[1]],
        'Positive Attrition Cases': [int(df_tree['Attrition'].sum()), int(df_nontree['Attrition'].sum())],
        'Negative Attrition Cases': [int((df_tree['Attrition'] == 0).sum()), int((df_nontree['Attrition'] == 0).sum())],
    }
)
summary

,Dataset,Rows,Columns,Positive Attrition Cases,Negative Attrition Cases
0,Tree-based,1191,35,515,676
1,Non-tree-based,1191,190,515,676


## Train and Evaluate Candidate Models
We compare four common baseline classifiers:

- Logistic Regression
- Support Vector Machine (RBF kernel)
- Decision Tree
- Random Forest

The train/test split uses 80 percent for training and 20 percent for testing with stratification on the target.

In [5]:
def evaluate_model(model_name, model, X, y, dataset_name):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        scores = model.decision_function(X_test)
        y_score = (scores - scores.min()) / (scores.max() - scores.min())

    return {
        'Model': model_name,
        'Dataset': dataset_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_score),
        'Confusion Matrix': confusion_matrix(y_test, y_pred).tolist(),
    }

X_nontree = df_nontree.drop(columns=['Attrition'])
y_nontree = df_nontree['Attrition']
X_tree = df_tree.drop(columns=['Attrition'])
y_tree = df_tree['Attrition']

results = [
    evaluate_model('Logistic Regression', LogisticRegression(max_iter=5000, random_state=42), X_nontree, y_nontree, 'Non-tree dataset'),
    evaluate_model('SVM (RBF)', SVC(kernel='rbf', probability=True, random_state=42), X_nontree, y_nontree, 'Non-tree dataset'),
    evaluate_model('Decision Tree', DecisionTreeClassifier(max_depth=6, min_samples_split=10, random_state=42), X_tree, y_tree, 'Tree dataset'),
    evaluate_model('Random Forest', RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_split=5, random_state=42, n_jobs=-1), X_tree, y_tree, 'Tree dataset'),
]

results_df = pd.DataFrame(results).sort_values(by=['F1 Score', 'Accuracy', 'Recall'], ascending=False)
results_df[['Model', 'Dataset', 'Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC']].round(4)

,Model,Dataset,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,Non-tree dataset,1.0000,1.0000,1.0000,1.0000,1.0000
2,Decision Tree,Tree dataset,0.9791,0.9804,0.9709,0.9756,0.9960
3,Random Forest,Tree dataset,0.9791,0.9900,0.9612,0.9754,0.9892
1,SVM (RBF),Non-tree dataset,0.9540,0.9894,0.9029,0.9442,0.9946


## Initial Metrics
The table above provides the initial metrics for the first implementation of each candidate model. Based on these results, Logistic Regression is selected as the strongest initial model.

In [6]:
best_model_name = results_df.iloc[0]['Model']
best_model_name

'Logistic Regression'

## Extra Validation for the Selected Model
To reduce the risk that the result is caused by a single train/test split, we also perform 5-fold stratified cross-validation on the selected model.

In [7]:
log_reg = LogisticRegression(max_iter=5000, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_validate(
    log_reg,
    X_nontree,
    y_nontree,
    cv=cv,
    scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'],
)

cv_summary = pd.DataFrame(
    {
        'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC'],
        'Mean CV Score': [
            cv_scores['test_accuracy'].mean(),
            cv_scores['test_precision'].mean(),
            cv_scores['test_recall'].mean(),
            cv_scores['test_f1'].mean(),
            cv_scores['test_roc_auc'].mean(),
        ],
    }
)
cv_summary.round(4)

,Metric,Mean CV Score
0,Accuracy,0.9958
1,Precision,1.0000
2,Recall,0.9903
3,F1 Score,0.9951
4,ROC-AUC,0.9999


## Conclusion
Logistic Regression is the recommended initial model for this project because it achieved the best overall performance on the prepared non-tree dataset. It is also a strong fit for binary classification, computationally efficient, and easier to explain in an academic report than more complex alternatives.

This makes it a suitable baseline for the final project stage, while tree-based models remain useful comparison benchmarks.